# this is an attempt in Julia to implement the BLP algorithm 

In [11]:
using Pkg 
using CSV
using DataFrames
using LinearAlgebra
using Statistics
using DataFrames
using Random
using Distributions
using Optim
include("functions.jl")

estimate_parameter (generic function with 1 method)

In [12]:
df = CSV.read("bem.csv", DataFrame)

df_sample = df[df[!, :market_id] .== 4,:]
print(df_sample)

6×10 DataFrame
 Row │ market_id  product_id  prod_char   price      share        true_delta  share_not_buy  instrument_cost  instrument_char  instrument_price 
     │ Int64      Int64       Float64     Float64    Float64      Float64     Float64        Float64          Float64          Float64          
─────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │         4           0   1.24846    -0.253603  0.0577142      3.45865      0.00181682         0.854996       -1.24846            0.253603
   2 │         4           1   0.312115    1.30929   4.65053e-5    -3.6652       0.00181682         0.47103        -0.312115          -1.30929
   3 │         4           2  -1.42681     0.261604  3.83106e-5    -3.85959      0.00181682        -0.796933        1.42681           -0.261604
   4 │         4           3   0.0445878  -1.86487   0.93897        6.24771      0.00181682        -1.83671        -0.0

In [13]:
function objective_function(
    sigma_guess,
    input_dataset,
    parameter_guess,
    tolerance,
    n_consumer_sim,
    instrument_choice)

    # Step 1: initial delta
    delta_array = mean_utility(input_dataset, sigma_guess, parameter_guess, tolerance, n_consumer_sim)
    
    # Step 2: linear IV estimation (alpha, beta)
    alpha_hat, beta_hat = estimate_parameter(input_dataset, instrument_choice, delta_array)

    # # Step 3: re-calculate delta using estimated parameters
    delta_array = mean_utility(input_dataset, sigma_guess, [alpha_hat, beta_hat], tolerance, n_consumer_sim)

    # print(delta_array)
    # # Step 4: compute residual
    x = input_dataset[:, :prod_char]
    p = input_dataset[:, :price]
    residual = delta_array .- (beta_hat .* x .+ alpha_hat .* p)

    # Step 5: compute GMM loss
    Z = Matrix(input_dataset[:, [instrument_choice, :price]])
    g = (Z' * residual) / length(residual)
    loss = dot(g, g)  # equivalent to g'T * g

    return loss
end
# return the loss value from the objective function 

objective_function (generic function with 1 method)

In [14]:
# optimize(x -> objective_function(sigma, df, para_guess, 0.05, 200, :instrument_char), [2.0], LBFGS())

function main(
    instrument_choice,
    nonlinear_guess,
    linear_guess,
    market_data,
    tolerance,
    n_consumer_sim
)
    println("Running optimization with instrument ", instrument_choice)

    result = optimize(
        sigma -> objective_function(sigma, market_data, linear_guess, tolerance, n_consumer_sim, :instrument_char),
        nonlinear_guess, # initial guess as Float64
        LBFGS()
    )

    sigma_estimated = Optim.minimizer(result)

    delta_final = mean_utility(market_data, sigma_estimated, linear_guess, tolerance, n_consumer_sim)
    linear_estimated = estimate_parameter(market_data, instrument_choice, delta_final)

    println("Estimated sigma (nonlinear): ", sigma_estimated)
    println("Estimated linear params: ", linear_estimated)
end

main(:instrument_char, [0.5], [1 4], df, 0.1, 200)

Running optimization with instrument instrument_char
Estimated sigma (nonlinear): [-3.5445919677128503]
Estimated linear params: (2.4288855016233506, 0.22075858220421288)
